# 06 · Scratch 2D CNN (early fusion) on cache C: Q2 development evaluation

## Purpose
The first deep model: a small CNN trained from scratch on the frozen representation C (six accelerometer
order-spectrograms + the aligned microphone log-Mel, 7 × 128 × 25 per 1 s window), evaluated for **Q2 (which known
fault?)** with the same folder-grouped development protocol as the classical baselines (notebook 05b), so the numbers
are directly comparable.

**Scope (CLAUDE.md §7, fixed before running).**
- **Q2 only.** These models are trained on all ten classes. Their scores and embeddings are for Q2 analysis only and are
  tagged *"Q2-only; trained on all 10 classes; not valid for LOCO"*. LOCO (Stage 3C) will retrain the encoder without the
  withheld class.
- **Early fusion only** (all channels into one network). CNN-7 vs CNN-6 (vibration only) is an early-fusion *preview*;
  late fusion stays on the roadmap and the fusion study is not complete.
- **One training seed.** The bootstrap intervals describe uncertainty from the evaluated recordings and folders, not
  variation between CNN training runs.

All numbers are **development findings** on the development pool; the reserved tests stay sealed.

## Inputs
`results/cnn/{cnn7,cnn6}_dev_train.json`, `results/cnn/dev_eval.json`, `results/cnn/dev_per_folder_accuracy.csv`,
`results/scores/{cnn7,cnn6}_dev_embeddings.npz` and the MLflow runs (`stage = 3A`), all written by
`scripts/06_train_cnn.py` (tasks `train`, `evaluate`).

<div style="background-color:#e3f2fd; color:#0d2538; padding:10px 14px; border-radius:6px; border-left:4px solid #64b5f6; margin:4px 0;"><b>What this code does:</b> Loads the settings, <code>metadata.csv</code> and the saved CNN results, stops if they were produced under different split assignments, and finds the Stage 3A training runs in MLflow.</div>

In [ ]:
import json

import matplotlib.pyplot as plt
import mlflow
import numpy as np
import pandas as pd
from sklearn.decomposition import PCA

from ml.cache import check_split_hash
from ml.config import load_config
from ml.labels import CLASS6, CLASS10
from ml.paths import DEFAULT_CONFIG, SYSTEM_CODE_DIR, mlflow_uri

cfg = load_config(DEFAULT_CONFIG)
meta = pd.read_csv(SYSTEM_CODE_DIR / "metadata.csv")
RES = SYSTEM_CODE_DIR / "results" / "cnn"
FIG = SYSTEM_CODE_DIR / "results" / "figures"
SCORES = SYSTEM_CODE_DIR / "results" / "scores"
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.3})
ev = json.loads((RES / "dev_eval.json").read_text())
train = {v: json.loads((RES / f"{v}_dev_train.json").read_text()) for v in ("cnn7", "cnn6")}
check_split_hash(ev["provenance"]["split_hash"], meta, "dev_eval.json")
for v, t in train.items():
    check_split_hash(t["provenance"]["split_hash"], meta, f"{v}_dev_train.json")
mlflow.set_tracking_uri(mlflow_uri(cfg))
runs = mlflow.search_runs(experiment_names=["machinesense"], filter_string="tags.stage = '3A'")
parents = {v: train[v]["provenance"]["mlflow_run_id"] for v in train}
print(ev["provenance"]["scope"])
print(ev["provenance"]["bootstrap"])
print("MLflow parent runs:", parents)

## 1. Architecture and fixed settings
Four convolution blocks (each: two 3 × 3 convolution – batch-norm – ReLU layers; the first three end in 2 × 2 max-pooling,
128 × 25 → 16 × 3), global average pooling, a 128-d embedding **z** (linear – batch-norm – ReLU), dropout 0.2 and a linear
head for 10 classes. Everything below was fixed in `configs/base.yaml` (`cnn`) before the first run: fixed epochs, no early
stopping, nothing chosen on the held-out fold.

<div style="background-color:#e3f2fd; color:#0d2538; padding:10px 14px; border-radius:6px; border-left:4px solid #64b5f6; margin:4px 0;"><b>What this code does:</b> Builds the two networks (7 input channels and 6 input channels), counts the trainable parameters of each part, and lists the fixed training settings from the config.</div>

In [ ]:
from ml.models.cnn import ScratchCNN, n_parameters

rows = []
for name, ch in (("CNN-7 (vibration + microphone)", 7), ("CNN-6 (vibration only)", 6)):
    m = ScratchCNN(ch, len(CLASS10), cfg.cnn)
    parts = {f"block {i + 1}": n_parameters(b) for i, b in enumerate(m.features)}
    parts.update({"embedding": n_parameters(m.embed), "head": n_parameters(m.head), "total": n_parameters(m)})
    rows.append(pd.Series(parts, name=name))
display(pd.DataFrame(rows))
c = cfg.cnn
pd.Series({"epochs (fixed)": c.epochs, "batch": c.batch_size, "optimiser": f"AdamW lr {c.lr}, weight decay {c.weight_decay}",
           "schedule": f"{c.warmup_epochs} epoch linear warm-up, then cosine to 0",
           "loss": "cross-entropy, class weights from the training windows",
           "augmentation (training only)": f"gain ±{c.gain_jitter_db} dB, circular shift ±{c.time_shift_frames} frames, "
                                            f"one time mask ≤ {c.time_mask_frames} frames; no frequency masking",
           "normalisation": "global per-channel mean/std from norm_stats (development mode, the fold's training recordings)",
           "precision": "float16 resident cache; float32 batches, weights and arithmetic; no AMP",
           "seed": c.seed}, name="setting").to_frame()

## 2. Training curves
Training loss and accuracy on the (augmented) training batches, per fold, read back from MLflow. No curve on held-out
data is shown or used: the number of epochs was fixed in advance.

<div style="background-color:#e3f2fd; color:#0d2538; padding:10px 14px; border-radius:6px; border-left:4px solid #64b5f6; margin:4px 0;"><b>What this code does:</b> Reads each fold's per-epoch training loss and training accuracy from MLflow and plots them, one line per fold, for CNN-7 and CNN-6. The loss axis is logarithmic.</div>

In [ ]:
client = mlflow.tracking.MlflowClient()
fig, axes = plt.subplots(2, 2, figsize=(11, 6.5), sharex=True)
for j, v in enumerate(("cnn7", "cnn6")):
    kids = runs[runs["tags.mlflow.parentRunId"] == parents[v]].sort_values("tags.fold")
    for _, r in kids.iterrows():
        for i, metric in enumerate(("train_loss", "train_acc")):
            h = client.get_metric_history(r.run_id, metric)
            axes[i, j].plot([m.step for m in h], [m.value for m in h], label=f"fold {r['tags.fold']}")
    axes[0, j].set(yscale="log", ylabel="training loss (weighted CE)", title=f"{v.upper()}: training loss")
    axes[1, j].set(xlabel="epoch", ylabel="training accuracy (windows)", ylim=(0.4, 1.01), title=f"{v.upper()}: training accuracy")
    axes[0, j].legend(fontsize=8)
fig.tight_layout()
fig.savefig(FIG / "06_cnn_training_curves.png")

## 3. Results per fold and pooled
Recording-level decisions use the same aggregation as the classical baselines: clip window probabilities, average their
logs over the 9 windows, renormalise, argmax; class6 by summing. Each pool recording is scored once, by its own fold's
model. Pooled intervals: the stratified mixed-unit bootstrap (fault folders resampled within class; normal recordings
individually), identical draws for every model. Logistic regression is the capped R1 reference refitted on the same folds.

<div style="background-color:#e3f2fd; color:#0d2538; padding:10px 14px; border-radius:6px; border-left:4px solid #64b5f6; margin:4px 0;"><b>What this code does:</b> Builds the per-fold table (macro-F1 and balanced accuracy for class10 and class6, per model and fold) and the pooled table with 95 % bootstrap intervals.</div>

In [ ]:
names = {"cnn7": "CNN-7", "cnn6": "CNN-6", "logreg": "LogReg (capped R1)"}
per_fold = pd.DataFrame([{"model": names[m], "fold": int(k), **v} for m, r in ev["models"].items()
                         for k, v in r["per_fold"].items()]).set_index(["model", "fold"])
display(per_fold.drop(columns="n_recordings").round(3).unstack("fold"))
fmt = lambda d, key: f"{d[key]:.3f} [{d[key + '_ci'][0]:.3f}, {d[key + '_ci'][1]:.3f}]"
pd.DataFrame([{"model": names[m], "class10 macro-F1": fmt(r["class10"], "macro_f1"),
               "class10 balanced acc.": fmt(r["class10"], "balanced_accuracy"),
               "class6 macro-F1": fmt(r["class6"], "macro_f1"), "class6 balanced acc.": fmt(r["class6"], "balanced_accuracy"),
               "recordings": r["class10"]["n"]} for m, r in ev["models"].items()]).set_index("model")

<div style="background-color:#e3f2fd; color:#0d2538; padding:10px 14px; border-radius:6px; border-left:4px solid #64b5f6; margin:4px 0;"><b>What this code does:</b> Shows per-class F1 (class10) for the three models side by side.</div>

In [ ]:
pd.DataFrame({names[m]: r["class10"]["per_class_f1"] for m, r in ev["models"].items()}).loc[list(CLASS10)].round(3)

## 4. Confusion matrices (class10, pooled over the three folds)

<div style="background-color:#e3f2fd; color:#0d2538; padding:10px 14px; border-radius:6px; border-left:4px solid #64b5f6; margin:4px 0;"><b>What this code does:</b> Draws the recording-level class10 confusion matrices of CNN-7 and CNN-6: rows are the true class, columns the predicted class, numbers are recordings. Off-diagonal cells are errors.</div>

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
short = [c.replace("_misalignment", "_mis").replace("underhang", "under").replace("overhang", "over") for c in CLASS10]
for ax, m in zip(axes, ("cnn7", "cnn6")):
    cm = np.array(ev["models"][m]["class10"]["confusion"])
    ax.imshow(np.log1p(cm), cmap="Blues")
    for i in range(len(CLASS10)):
        for j in range(len(CLASS10)):
            if cm[i, j]:
                ax.text(j, i, cm[i, j], ha="center", va="center", fontsize=7, color="w" if i == j else "k")
    ax.set_xticks(range(10), short, rotation=60, ha="right", fontsize=7)
    ax.set_yticks(range(10), short, fontsize=7)
    ax.set(xlabel="predicted", ylabel="true", title=f"{names[m]}: recordings, class10")
    ax.grid(False)
fig.tight_layout()
fig.savefig(FIG / "06_cnn_confusion.png")

## 5. Hardest folders
Per-folder accuracy (a folder = class × severity, a proxy for a session). In the classical evaluation (05b) the 20 g
bearing folders were the hardest.

<div style="background-color:#e3f2fd; color:#0d2538; padding:10px 14px; border-radius:6px; border-left:4px solid #64b5f6; margin:4px 0;"><b>What this code does:</b> Lists the folders with the lowest CNN-7 accuracy, with CNN-6 and logistic regression next to them and the number of recordings per folder.</div>

In [ ]:
fa = pd.read_csv(RES / "dev_per_folder_accuracy.csv", index_col=0).rename(columns=names)
fa.sort_values("CNN-7").head(10).round(3)

## 6. Paired comparisons
Differences in macro-F1 over identical bootstrap draws; P(diff > 0) is the share of draws in which the first model is
better. No winner is claimed without a paired interval that excludes 0. With one CNN seed, the intervals do not cover
training-run variation.

<div style="background-color:#e3f2fd; color:#0d2538; padding:10px 14px; border-radius:6px; border-left:4px solid #64b5f6; margin:4px 0;"><b>What this code does:</b> Tabulates the paired macro-F1 differences (class10 and class6) with their 95 % intervals.</div>

In [ ]:
pd.DataFrame([{"comparison": k.replace("_minus_", " − "), "view": v, "difference": d[v]["diff"],
               "95 % interval": f"[{d[v]['ci'][0]:+.3f}, {d[v]['ci'][1]:+.3f}]", "P(diff > 0)": d[v]["p_positive"]}
              for k, d in ev["paired"].items() for v in ("class10", "class6")]).set_index(["comparison", "view"]).round(3)

## 7. Embedding PCA, one per fold model
Each fold's CNN has its own embedding space, so embeddings from different fold models are never combined: one PCA per
fold model, fitted only on the windows that model scored (its held-out fold), coloured by true class.

<div style="background-color:#e3f2fd; color:#0d2538; padding:10px 14px; border-radius:6px; border-left:4px solid #64b5f6; margin:4px 0;"><b>What this code does:</b> For each variant and fold, takes the 128-d embeddings of the windows scored by that fold's model only, checks they all come from one run, fits a separate 2-component PCA to them and plots the windows coloured by class.</div>

In [ ]:
colors = dict(zip(CLASS10, plt.cm.tab10(np.arange(10))))
fig, axes = plt.subplots(2, 3, figsize=(15, 9))
for i, v in enumerate(("cnn7", "cnn6")):
    z = np.load(SCORES / f"{v}_dev_embeddings.npz")
    assert str(z["scope"]) == ev["provenance"]["scope"]
    lab = pd.read_parquet(SCORES / f"{v}_dev.parquet", columns=["rec_id", "window_idx", "y_true"])
    assert (lab.rec_id.to_numpy() == z["rec_id"]).all() and (lab.window_idx.to_numpy() == z["window_idx"]).all()
    for k in range(3):
        sel = z["dev_fold"] == k
        assert len(set(z["run_id"][sel])) == 1  # one fold model per PCA
        pca = PCA(n_components=2).fit(z["embedding"][sel])
        p = pca.transform(z["embedding"][sel])
        ax = axes[i, k]
        for c in CLASS10:
            m = lab.y_true.to_numpy()[sel] == c
            ax.scatter(p[m, 0], p[m, 1], s=4, alpha=0.5, color=colors[c], label=c)
        ev_ = pca.explained_variance_ratio_
        ax.set(title=f"{v.upper()} fold {k} model (own PCA)", xlabel=f"PC1 ({ev_[0]:.0%})", ylabel=f"PC2 ({ev_[1]:.0%})")
axes[0, 2].legend(fontsize=7, markerscale=3, loc="best")
fig.tight_layout()
fig.savefig(FIG / "06_cnn_embedding_pca.png")

## 8. Training time and CPU inference latency
Training on MPS (Apple M4 Pro); latency is CPU time to score one recording (9 windows in one batch) from its cached
representation, after the model is moved to the CPU. It excludes building the representation (≈ 0.51 s per recording,
Stage 2C-2).

<div style="background-color:#e3f2fd; color:#0d2538; padding:10px 14px; border-radius:6px; border-left:4px solid #64b5f6; margin:4px 0;"><b>What this code does:</b> Tabulates, per variant and fold, the training seconds, the number of training windows and the median CPU milliseconds per recording, plus the smoke-run step time and memory.</div>

In [ ]:
display(pd.DataFrame([{"variant": v, "fold": f["fold"], "train windows": f["n_train_windows"],
                       "train s (MPS)": round(f["train_seconds"], 1), "CPU ms / recording": round(f["cpu_ms_per_recording"], 2)}
                      for v, t in train.items() for f in t["folds"]]).set_index(["variant", "fold"]))
smoke = json.loads((RES / "smoke.json").read_text())
pd.Series({"median step (ms)": round(smoke["median_step_ms"], 1), "MPS allocated (MB)": round(smoke["memory"]["current_allocated_mb"]),
           "MPS driver (MB)": round(smoke["memory"]["driver_allocated_mb"]),
           "projected min, both variants": round(smoke["projection"]["train_minutes_both_variants"], 1)}, name="smoke run").to_frame()

## Findings
Development findings on the 804-recording pool (K = 3 folder-grouped folds); not reserved-test results; one training seed.
- **Both CNNs are ahead of the classical reference on class10.** Macro-F1 with the mixed-unit bootstrap interval:
  CNN-7 0.980 [0.966, 0.993], CNN-6 0.989 [0.980, 0.998], logistic regression (capped R1, refitted on the same folds)
  0.949 [0.908, 0.983]. Paired: CNN-7 − LR +0.031 [+0.006, +0.062]; CNN-6 − LR +0.041 [+0.010, +0.078]. On class6 the
  intervals include 0 (CNN-7 − LR +0.008 [−0.011, +0.029]).
- **Early-fusion preview: adding the microphone did not help here.** CNN-7 − CNN-6 = −0.009 [−0.019, −0.001] on class10
  (P(diff > 0) = 0.01). This is one seed and one fusion design; it says nothing yet about late fusion, and a
  one-seed difference of this size may be within run-to-run training variation (not measured).
- **The 20 g bearing folders are still the hardest**, but less so: underhang cage 20 g 0.84 (CNN-7 and CNN-6; LR 0.64),
  overhang cage 20 g 0.85 / 0.96 (LR 0.69), overhang outer 20 g 0.90 / 1.00 (LR 0.57). Vertical misalignment 0.63 mm is
  the one folder where both CNNs (0.88 / 0.92) are below LR (1.00).
- **Training fits the training windows completely** (training accuracy 1.0 after about 6 epochs, final weighted loss
  ≈ 6 × 10⁻⁴ in every fold); the 40 epochs were fixed in advance and the held-out folds did not influence them.
- **Cost:** about 60 s of MPS training per fold (4,700–4,900 windows, 19 steps/epoch, ≈ 76 ms/step); CPU inference
  ≈ 4 ms per recording from the cached representation (plus ≈ 0.51 s to build the representation).
- **Embeddings (one PCA per fold model):** classes form compact clusters in every fold model; the cage and ball bearing
  clusters sit closest together, in line with where the errors are. These embeddings are Q2-only and must not be reused
  for LOCO or Q3.


## Outputs written
- `results/figures/06_cnn_training_curves.png`, `06_cnn_confusion.png`, `06_cnn_embedding_pca.png`
- (by `scripts/06_train_cnn.py`) `results/cnn/*.json|csv`; `results/scores/{cnn7,cnn6}_dev.parquet` and
  `_embeddings.npz` (Q2-only); `artifacts/cnn/<variant>_dev/fold<k>.pt`; MLflow runs tagged `stage = 3A`.